# 03. PySpark DataFrame Basics & Schema Definition

In this notebook, we move from RDDs to the modern standard of PySpark: **DataFrames**.

### Topics Covered:
1. **What is a DataFrame in PySpark?**
2. **Creating DataFrames from Python lists, RDDs, and Pandas**
3. **Understanding Schemas: Inferred vs. Explicit (`StructType` & `StructField`)**
4. **Why explicit schemas are essential for production**
5. **Inspecting DataFrames (`show`, `printSchema`, `describe`, `dtypes`)**
6. **Selecting and Renaming Columns (`select`, `col`, `alias`, `withColumnRenamed`)**
7. **Adding, Modifying, and Casting Columns (`withColumn`, `cast`)**
8. **Stopping the SparkSession to free memory**


---
## 1. What is a PySpark DataFrame?

A PySpark DataFrame is a distributed collection of data organized into named columns.
Think of it as:
$$\text{DataFrame} = \text{Distributed RDD} + \text{Schema (Column Names & Types)} + \text{Catalyst Query Optimizer}$$

### Key Advantages:
* **Structured**: Spark knows column names and data types.
* **Catalyst Optimizer**: Spark automatically reorganizes your code to execute in the fastest possible order.
* **Tungsten Engine**: Stores data in compact binary format in off-heap memory, bypassing Java garbage collection overhead.


---
## 2. Initializing SparkSession


In [1]:


from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from pyspark.sql.types import (
    StructType, StructField, 
    StringType, IntegerType, DoubleType, BooleanType, DateType
)

# Start SparkSession
spark = SparkSession.builder \
    .appName("03_DataFrame_Basics") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession initialized!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/05 18:31:49 WARN Utils: Your hostname, Tiger, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/05 18:31:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 18:31:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession initialized!


---
## 3. Creating DataFrames

### Method A: From a Python List of Tuples (Automatic / Inferred Schema)


In [3]:
users_data = [
    (1, "Alice Smith", 28, "New York", 75000.0, True),
    (2, "Bob Johnson", 35, "Chicago", 82000.5, False),
    (3, "Charlie Lee", 42, "San Francisco", 120000.0, True),
    (4, "Diana Prince", 31, "Austin", 95000.0, True),
    (5, "Evan Wright", 24, "Seattle", 61000.0, False)
]

columns = ["id", "full_name", "age", "city", "salary", "is_active"]

df_inferred = spark.createDataFrame(users_data, columns)
df_inferred.show()
df_inferred.printSchema()


+---+------------+---+-------------+--------+---------+
| id|   full_name|age|         city|  salary|is_active|
+---+------------+---+-------------+--------+---------+
|  1| Alice Smith| 28|     New York| 75000.0|     true|
|  2| Bob Johnson| 35|      Chicago| 82000.5|    false|
|  3| Charlie Lee| 42|San Francisco|120000.0|     true|
|  4|Diana Prince| 31|       Austin| 95000.0|     true|
|  5| Evan Wright| 24|      Seattle| 61000.0|    false|
+---+------------+---+-------------+--------+---------+

root
 |-- id: long (nullable = true)
 |-- full_name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- is_active: boolean (nullable = true)



---
## 4. Defining Explicit Schemas (`StructType` & `StructField`)

### Why should you define explicit schemas instead of inferring?
1. **Performance**: Inferring schema requires Spark to scan data beforehand to guess types. For terabytes of data, this is very slow.
2. **Reliability**: Guesses can be wrong (e.g. a column with all integers except one string might crash or corrupt data).
3. **Data Quality**: Explicit schemas reject or flag corrupted records early.


In [4]:
# Explicit schema definition
user_schema = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("full_name", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=True),
    StructField("city", StringType(), nullable=True),
    StructField("salary", DoubleType(), nullable=True),
    StructField("is_active", BooleanType(), nullable=True)
])

# Create DataFrame using the explicit schema
df = spark.createDataFrame(users_data, schema=user_schema)

print("DataFrame created with explicit schema:")
df.show()
df.printSchema()


DataFrame created with explicit schema:
+---+------------+---+-------------+--------+---------+
| id|   full_name|age|         city|  salary|is_active|
+---+------------+---+-------------+--------+---------+
|  1| Alice Smith| 28|     New York| 75000.0|     true|
|  2| Bob Johnson| 35|      Chicago| 82000.5|    false|
|  3| Charlie Lee| 42|San Francisco|120000.0|     true|
|  4|Diana Prince| 31|       Austin| 95000.0|     true|
|  5| Evan Wright| 24|      Seattle| 61000.0|    false|
+---+------------+---+-------------+--------+---------+

root
 |-- id: integer (nullable = false)
 |-- full_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- is_active: boolean (nullable = true)



---
## 5. Inspecting DataFrames

PySpark provides several methods to examine your DataFrame structure and contents.


In [7]:
# 1. show(n, truncate): Preview rows
# truncate=False ensures long text is not cut off
df.show(3, truncate=False)

# 2. columns: List of column names
print("Column names:", df.columns)

# 3. dtypes: List of (column_name, data_type) tuples
print("Data types:", df.dtypes)

# 4. count(): Number of rows
print("Total rows:", df.count())

# 5. describe(): Summary statistics (count, mean, stddev, min, max) for numeric columns
print("Statistical summary:")
df.describe(["age", "salary"]).show()


+---+-----------+---+-------------+--------+---------+
|id |full_name  |age|city         |salary  |is_active|
+---+-----------+---+-------------+--------+---------+
|1  |Alice Smith|28 |New York     |75000.0 |true     |
|2  |Bob Johnson|35 |Chicago      |82000.5 |false    |
|3  |Charlie Lee|42 |San Francisco|120000.0|true     |
+---+-----------+---+-------------+--------+---------+
only showing top 3 rows
Column names: ['id', 'full_name', 'age', 'city', 'salary', 'is_active']
Data types: [('id', 'int'), ('full_name', 'string'), ('age', 'int'), ('city', 'string'), ('salary', 'double'), ('is_active', 'boolean')]
Total rows: 5
Statistical summary:
+-------+-----------------+------------------+
|summary|              age|            salary|
+-------+-----------------+------------------+
|  count|                5|                 5|
|   mean|             32.0|           86600.1|
| stddev|6.892024376045111|22344.996085253628|
|    min|               24|           61000.0|
|    max|         

---
## 6. Selecting and Renaming Columns

You can select columns using:
* Column name string: `"age"`
* `col()` function: `col("age")` (recommended because it enables expressions)
* DataFrame reference: `df["age"]` or `df.age`


In [8]:
# Selecting specific columns
df.select("full_name", "city", "salary").show(3)

# Renaming while selecting using alias()
df.select(
    col("full_name").alias("employee_name"),
    col("salary").alias("annual_income")
).show(3)

# Renaming an existing column in-place with withColumnRenamed()
renamed_df = df.withColumnRenamed("city", "location")
renamed_df.show(3)


+-----------+-------------+--------+
|  full_name|         city|  salary|
+-----------+-------------+--------+
|Alice Smith|     New York| 75000.0|
|Bob Johnson|      Chicago| 82000.5|
|Charlie Lee|San Francisco|120000.0|
+-----------+-------------+--------+
only showing top 3 rows
+-------------+-------------+
|employee_name|annual_income|
+-------------+-------------+
|  Alice Smith|      75000.0|
|  Bob Johnson|      82000.5|
|  Charlie Lee|     120000.0|
+-------------+-------------+
only showing top 3 rows
+---+-----------+---+-------------+--------+---------+
| id|  full_name|age|     location|  salary|is_active|
+---+-----------+---+-------------+--------+---------+
|  1|Alice Smith| 28|     New York| 75000.0|     true|
|  2|Bob Johnson| 35|      Chicago| 82000.5|    false|
|  3|Charlie Lee| 42|San Francisco|120000.0|     true|
+---+-----------+---+-------------+--------+---------+
only showing top 3 rows


---
## 7. Adding and Modifying Columns (`withColumn`)

`withColumn(colName, colExpression)` is one of the most frequently used methods in PySpark.
* If `colName` already exists, it updates it.
* If `colName` does not exist, it adds it as a new column.


In [9]:
# 1. Add a calculated column: Monthly Salary (salary / 12)
df_with_monthly = df.withColumn("monthly_salary", col("salary") / 12)

# 2. Add a static/constant column using lit()
df_with_company = df_with_monthly.withColumn("company", lit("TechCorp Inc"))

# 3. Add a boolean column: Is High Earner? (salary >= 90000)
df_transformed = df_with_company.withColumn("is_high_earner", col("salary") >= 90000)

df_transformed.select("full_name", "salary", "monthly_salary", "is_high_earner", "company").show()


+------------+--------+-----------------+--------------+------------+
|   full_name|  salary|   monthly_salary|is_high_earner|     company|
+------------+--------+-----------------+--------------+------------+
| Alice Smith| 75000.0|           6250.0|         false|TechCorp Inc|
| Bob Johnson| 82000.5|         6833.375|         false|TechCorp Inc|
| Charlie Lee|120000.0|          10000.0|          true|TechCorp Inc|
|Diana Prince| 95000.0|7916.666666666667|          true|TechCorp Inc|
| Evan Wright| 61000.0|5083.333333333333|         false|TechCorp Inc|
+------------+--------+-----------------+--------------+------------+



---
## 8. Casting Column Types (`cast`)

You often need to convert data types (e.g., string to integer, or integer to double).


In [10]:
# Example: casting 'age' from Integer to Double, and 'id' to String
converted_df = df.withColumn("age_double", col("age").cast(DoubleType())) \
                 .withColumn("id_str", col("id").cast("string"))

converted_df.printSchema()


root
 |-- id: integer (nullable = false)
 |-- full_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- age_double: double (nullable = true)
 |-- id_str: string (nullable = false)



---
## 9. Dropping Columns

You can remove columns you no longer need using `.drop()`.


In [11]:
# Dropping single or multiple columns
clean_df = df_transformed.drop("company", "monthly_salary")
print("Remaining columns after drop:")
clean_df.printSchema()


Remaining columns after drop:
root
 |-- id: integer (nullable = false)
 |-- full_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- is_high_earner: boolean (nullable = true)



---
## 10. Summary of Notebook 03
* A DataFrame combines distributed RDDs with structured column schemas and Catalyst optimization.
* Always use **explicit schemas** (`StructType`, `StructField`) in production for speed and reliability.
* Inspect data with `show()`, `printSchema()`, `describe()`.
* Select and transform columns with `select()`, `col()`, `withColumn()`, `withColumnRenamed()`, `cast()`, and `drop()`.

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession
Always stop the session to release RAM and CPU resources.


In [12]:
# Stop SparkSession to free memory
spark.stop()
print("SparkSession stopped successfully! RAM released.")


SparkSession stopped successfully! RAM released.
